# 02. 손실함수와 경사하강법 — 직접 굴려보기

**대응 강의:** [3강 학습의 본질](../../Lecture/03-머신러닝-기본개념/03-학습의-본질.md)

이 노트북에서 배우는 것:
- 손실함수(MSE)의 **'골짜기' 모양**을 직접 그려본다
- 경사하강법을 **라이브러리 없이 직접 구현**해 한 걸음씩 바닥으로 내려가는 과정을 본다
- **학습률(learning rate)** 을 바꿔 수렴 / 너무 느림 / 발산을 직접 만든다

> '학습'이라는 추상적 단어의 정체를 코드로 분해합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 간단한 가짜 데이터: 진짜 관계는 y = 3x (기울기 3) + 약간의 노이즈
rng = np.random.default_rng(0)
x = np.linspace(-2, 2, 50)
true_w = 3.0
y = true_w * x + rng.normal(0, 1.0, size=x.shape)

plt.scatter(x, y, alpha=0.6)
plt.title('데이터 (숨은 진짜 기울기 = 3)')
plt.xlabel('x'); plt.ylabel('y'); plt.show()

## 1. 손실함수의 '골짜기' 그리기

모델을 `y = w * x` 로 단순화하고, **파라미터 `w` 하나**만 학습한다고 해봅시다.
`w`를 여러 값으로 바꿔가며 MSE 손실을 계산하면, 손실이 `w`의 함수임을 볼 수 있습니다.

$$\text{MSE}(w) = \frac{1}{n}\sum_i (y_i - w\,x_i)^2$$

In [ ]:
def mse_loss(w):
    pred = w * x
    return np.mean((y - pred) ** 2)

w_grid = np.linspace(-2, 8, 200)
losses = [mse_loss(w) for w in w_grid]

plt.plot(w_grid, losses)
plt.axvline(true_w, color='g', ls='--', label='진짜 w=3 (골짜기 바닥 근처)')
plt.xlabel('파라미터 w'); plt.ylabel('손실 MSE')
plt.title('손실함수의 골짜기 — 우리는 이 바닥을 찾고 싶다')
plt.legend(); plt.show()

## 2. 경사하강법 직접 구현

업데이트 규칙: $w \leftarrow w - \eta \cdot \nabla L(w)$

MSE의 기울기(그래디언트)는 미분으로 구하면:
$$\nabla L(w) = \frac{d}{dw}\,\frac{1}{n}\sum (y_i - w x_i)^2 = -\frac{2}{n}\sum x_i (y_i - w x_i)$$

In [ ]:
def gradient(w):
    pred = w * x
    return -2 * np.mean(x * (y - pred))   # MSE의 w에 대한 미분

def gradient_descent(lr=0.1, steps=30, w_start=-1.0):
    w = w_start
    history = [w]
    for _ in range(steps):
        w = w - lr * gradient(w)     # ★ 핵심 한 줄: 기울기 반대로 한 걸음
        history.append(w)
    return np.array(history)

path = gradient_descent(lr=0.1, steps=30, w_start=-1.0)
print('시작 w =', round(path[0], 3))
print('최종 w =', round(path[-1], 3), ' (진짜 값 3에 수렴!)')

In [ ]:
# 경사하강이 골짜기를 따라 내려가는 과정을 시각화
plt.plot(w_grid, losses, alpha=0.5, label='손실 골짜기')
plt.scatter(path, [mse_loss(w) for w in path], c=range(len(path)),
            cmap='autumn_r', s=40, zorder=3, label='경사하강 한 걸음씩')
plt.plot(path, [mse_loss(w) for w in path], 'k-', alpha=0.3)
plt.xlabel('파라미터 w'); plt.ylabel('손실')
plt.title('한 걸음씩 골짜기 바닥으로 (밝은색=나중 단계)')
plt.legend(); plt.show()

## 3. 학습률(η) 실험 — 가장 중요한 손잡이

같은 코드에서 **학습률만** 바꿔봅니다. 너무 작으면 굼뜨고, 너무 크면 튕겨나가 발산합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
settings = [(0.005, '너무 작음: 굼뜸'), (0.1, '적절: 안정 수렴'), (0.55, '너무 큼: 발산!')]

for ax, (lr, title) in zip(axes, settings):
    p = gradient_descent(lr=lr, steps=30, w_start=-1.0)
    p = np.clip(p, w_grid.min(), w_grid.max())   # 그래프 범위로 자르기
    ax.plot(w_grid, losses, alpha=0.4)
    ax.plot(p, [mse_loss(w) for w in p], 'o-', ms=4)
    ax.set_title(f'lr={lr}\n{title}')
    ax.set_xlabel('w'); ax.set_ylabel('손실')
plt.tight_layout(); plt.show()

print('발산 케이스의 w 변화:', gradient_descent(lr=0.55, steps=8).round(2))

## 정리

- **손실함수** = 파라미터의 함수이고, 그 모양은 '골짜기'다.
- **경사하강법** = 기울기 반대 방향으로 한 걸음씩 = `w ← w - η·∇L`
- **학습률**이 너무 작으면 느리고, 너무 크면 발산한다.

> 💡 이 루프(예측→손실→기울기→이동)가 선형회귀부터 신경망까지 거의 모든 학습의 골격입니다.

## 🎯 직접 해보기

1. `lr`을 0.55와 0.1 사이(예: 0.3, 0.45)로 바꿔가며 '발산 직전'의 경계를 찾아보세요.
2. `w_start`를 7.0(반대편 언덕)에서 시작해도 같은 바닥으로 수렴하나요?
3. (도전) 모델을 `y = w*x + b`로 바꿔 파라미터를 **두 개**(w, b) 학습하도록 확장해 보세요. 기울기를 w, b 각각에 대해 계산하면 됩니다.